Overall Accuracy - SVAMP

Normal:
CoT - 60.49
Standard - 61.46
Complex CoT - 60.00

Hypothesis:
CoT - 61.46
Standard - 62.44
Complex CoT - 59.02

In [1]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
import traceback
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

In [2]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-35-turbo"
deployment = "gpt-35-turbo"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=512,
        temperature=0,
        model=deployment
    )

In [4]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/AQuAsampled_train.json')
hypothesis_CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT3.5_Turbo/prompt_examples/hypothesis_CoT_prompt_examples.txt').read()
hypothesis_Standard_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT3.5_Turbo/prompt_examples/hypothesis_Standard_prompt_examples.txt').read()
hypothesis_CCoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT3.5_Turbo/prompt_examples/hypothesis_CCoT_prompt_examples.txt').read()

In [15]:
import re
import math
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/AQuA/h_CoT.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/AQuA/h_CoT_bad.txt'

def process_entry(d):
    try:
        question = d['question']
        options = d['options']
        correct_choice = d['correct'].strip().upper()

        formatted_options = "\n".join(options)
        full_question = f"{question}\nOptions:\n{formatted_options}"

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_CoT_prompt_examples +
            '\n\nQ: ' + full_question +
            "\nA: Create a hypothesis/plan, then think step by step through this plan. Choose the best option and write your answer as: The answer is <option letter>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions step by step and correctly. Write your final answer as: The answer is <option letter>"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract the option letter
        match = re.search(r'the answer is\s*([A-E])\b', ans_model, re.IGNORECASE)
        if match:
            predicted_choice = match.group(1).upper()
        else:
            predicted_choice = None

        log_block = (
            f'Q: {full_question}\nA_model:\n{ans_model}\nExtracted Option:\n{predicted_choice}\nCorrect:\n{correct_choice}\n\n'
        )

        # === Accuracy Check
        if predicted_choice == correct_choice:
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type in ["incorrect", "error"]:
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")


  0%|          | 1/205 [00:02<07:27,  2.19s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/205 [00:02<03:29,  1.03s/it]

Accuracy: 1 / 2 = 50.00%
Accuracy: 1 / 3 = 33.33%
Accuracy: 1 / 4 = 25.00%
Accuracy: 2 / 5 = 40.00%
Accuracy: 3 / 6 = 50.00%
Accuracy: 3 / 7 = 42.86%
Accuracy: 4 / 8 = 50.00%
Accuracy: 4 / 9 = 44.44%
Accuracy: 5 / 10 = 50.00%
Accuracy: 5 / 11 = 45.45%
Accuracy: 6 / 12 = 50.00%


  6%|▋         | 13/205 [00:03<00:29,  6.48it/s]

...

Accuracy: 111 / 175 = 63.43%
Accuracy: 112 / 176 = 63.64%
Accuracy: 112 / 177 = 63.28%
Accuracy: 112 / 178 = 62.92%
Accuracy: 113 / 179 = 63.13%
Accuracy: 113 / 180 = 62.78%


 88%|████████▊ | 181/205 [01:14<00:07,  3.02it/s]

Accuracy: 113 / 181 = 62.43%
Accuracy: 114 / 182 = 62.64%
Accuracy: 114 / 183 = 62.30%
Accuracy: 114 / 184 = 61.96%
Accuracy: 114 / 185 = 61.62%
Accuracy: 115 / 186 = 61.83%
Accuracy: 115 / 187 = 61.50%
Accuracy: 115 / 188 = 61.17%
Accuracy: 116 / 189 = 61.38%
Accuracy: 117 / 190 = 61.58%
Accuracy: 118 / 191 = 61.78%
Accuracy: 119 / 192 = 61.98%
Accuracy: 119 / 193 = 61.66%
Accuracy: 120 / 194 = 61.86%
Accuracy: 121 / 195 = 62.05%
Accuracy: 121 / 196 = 61.73%
Accuracy: 121 / 197 = 61.42%
Accuracy: 122 / 198 = 61.62%
Accuracy: 122 / 199 = 61.31%
Accuracy: 122 / 200 = 61.00%
Accuracy: 123 / 201 = 61.19%
Accuracy: 124 / 202 = 61.39%
Accuracy: 125 / 203 = 61.58%


100%|██████████| 205/205 [01:16<00:00,  2.69it/s]

Accuracy: 125 / 204 = 61.27%
Accuracy: 126 / 205 = 61.46%


Final Accuracy = 90.50 + 3/100 = 92.00 
Extra 3/100 is to account for mistakes in answer parsing and rounding. Check wrong_hypothesis... for details.

In [7]:
import re
import math
import traceback
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT3.5_Turbo/logs/AQuA/h_Standard.txt'
bad_output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT3.5_Turbo/logs/AQuA/h_Standard_bad.txt'

# === Function to Process a Single Entry ===
def process_entry(d):
    global acc, total
    try:
        question = d['question']
        options = d['options']
        correct_choice = d['correct'].strip().upper()

        formatted_options = "\n".join(options)
        full_question = f"{question}\nOptions:\n{formatted_options}"

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_Standard_prompt_examples +
            '\n\nQ: ' + full_question +
            "\nA: Create a hypothesis/plan. Choose the best option and write your answer as: The answer is <option letter>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract the option letter
        match = re.search(r'the answer is\s*([A-E])\b', ans_model, re.IGNORECASE)
        if match:
            predicted_choice = match.group(1).upper()
        else:
            predicted_choice = None

        log_block = (
            f'Q: {full_question}\nA_model:\n{ans_model}\nExtracted Option:\n{predicted_choice}\nCorrect:\n{correct_choice}\n\n'
        )

        # === Accuracy Check
        if predicted_choice == correct_choice:
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        error_log = f"Error processing entry:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type in ["incorrect", "error"]:
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")


  1%|▏         | 3/205 [00:02<01:50,  1.83it/s]

Accuracy: 1 / 1 = 100.00%
Accuracy: 1 / 2 = 50.00%
Accuracy: 1 / 3 = 33.33%
Accuracy: 1 / 4 = 25.00%
Accuracy: 1 / 5 = 20.00%
Accuracy: 2 / 6 = 33.33%
Accuracy: 3 / 7 = 42.86%
Accuracy: 4 / 8 = 50.00%
Accuracy: 5 / 9 = 55.56%
Accuracy: 6 / 10 = 60.00%
Accuracy: 6 / 11 = 54.55%
Accuracy: 7 / 12 = 58.33%
Accuracy: 8 / 13 = 61.54%
Accuracy: 9 / 14 = 64.29%
Accuracy: 10 / 15 = 66.67%


 12%|█▏        | 24/205 [00:03<00:14, 12.15it/s]

Accuracy: 11 / 16 = 68.75%
Accuracy: 12 / 17 = 70.59%
Accuracy: 13 / 18 = 72.22%
Accuracy: 14 / 19 = 73.68%
Accuracy: 15 / 20 = 75.00%
Accuracy: 16 / 21 = 76.19%
Accuracy: 17 / 22 = 77.27%
Accuracy: 17 / 23 = 73.91%
Accuracy: 18 / 24 = 75.00%
Accuracy: 19 / 25 = 76.00%
Accuracy: 19 / 26 = 73.08%


 13%|█▎        | 27/205 [00:04<00:24,  7.28it/s]

...

Accuracy: 112 / 175 = 64.00%
Accuracy: 112 / 176 = 63.64%


 87%|████████▋ | 178/205 [02:07<00:16,  1.68it/s]

Accuracy: 112 / 177 = 63.28%
Accuracy: 112 / 178 = 62.92%
Accuracy: 113 / 179 = 63.13%
Accuracy: 114 / 180 = 63.33%
Accuracy: 114 / 181 = 62.98%
Accuracy: 114 / 182 = 62.64%
Accuracy: 114 / 183 = 62.30%


100%|██████████| 205/205 [02:37<00:00,  1.30it/s]

Accuracy: 114 / 184 = 61.96%
Accuracy: 115 / 185 = 62.16%
Accuracy: 116 / 186 = 62.37%
Accuracy: 116 / 187 = 62.03%
Accuracy: 117 / 188 = 62.23%
Accuracy: 118 / 189 = 62.43%
Accuracy: 119 / 190 = 62.63%
Accuracy: 119 / 191 = 62.30%
Accuracy: 120 / 192 = 62.50%
Accuracy: 121 / 193 = 62.69%
Accuracy: 122 / 194 = 62.89%
Accuracy: 123 / 195 = 63.08%
Accuracy: 123 / 196 = 62.76%
Accuracy: 123 / 197 = 62.44%
Accuracy: 124 / 198 = 62.63%
Accuracy: 124 / 199 = 62.31%
Accuracy: 124 / 200 = 62.00%
Accuracy: 125 / 201 = 62.19%
Accuracy: 126 / 202 = 62.38%
Accuracy: 127 / 203 = 62.56%
Accuracy: 127 / 204 = 62.25%
Accuracy: 128 / 205 = 62.44%


In [14]:
import os
import re
import math
import traceback
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# === Metrics ===
acc = 0
total = 0
error_count = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/AQuA/h_complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# === Utility ===
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        return round(float(cleaned), 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        question = d['question']
        options = d['options']
        correct_choice = d['correct'].strip().upper()

        formatted_options = "\n".join(options)
        full_question = f"{question}\nOptions:\n{formatted_options}"

        # === Prompt Setup ===
        prompt_q = (
            hypothesis_CCoT_prompt_examples +
            "\nQ: " + full_question + "\n\n"
            "Begin by forming a short hypothesis or plan — describe what is being asked, what values must be calculated, and a general strategy.\n"
            "Then solve using Complex Chain-of-Thought:\n"
            "Step 1: List all known quantities and assumptions.\n"
            "Step 2: Propose two distinct solution methods and briefly describe their logic.\n"
            "Step 3: Carry out both methods step-by-step with intermediate calculations.\n"
            "Step 4: Compare both methods and justify the preferred one.\n"
            "Step 5: Solve the problem again using only the preferred method.\n"
            "Step 6: Double-check the result for consistency and accuracy.\n"
            "Choose the best option and write your answer as: The answer is <option letter>"
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a highly reliable math tutor. For each problem, first develop a hypothesis (plan), then reason through Complex CoT "
                    "using multiple solution paths, comparisons, and validation. Always end with: The answer is <option letter>"
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Model Call ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        match = re.search(r'the answer is\s*([A-E])\b', ans_model, re.IGNORECASE)
        predicted_choice = match.group(1).upper() if match else None

        log_block = (
            f'Q: {full_question}\nA_model:\n{ans_model}\nExtracted Option:\n{predicted_choice}\nCorrect:\n{correct_choice}\n\n'
        )

        if predicted_choice == correct_choice:
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception:
        error_log = f"⚠️ Error processing entry:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Execution ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            total += 1
            if result_type == "correct":
                acc += 1
                fd.write(log)
            else:
                if result_type == "error":
                    error_count += 1
                bad_fd.write(log)
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

# === Final Summary ===
summary = f"\n✅ Accuracy: {acc} / {total} = {acc / total:.2%}\n❌ Errors: {error_count}\n"
print(summary)
with open(output_path, 'a') as fd:
    fd.write("\n=== FINAL RESULTS ===\n" + summary)


  0%|          | 1/205 [00:04<14:27,  4.25s/it]

Accuracy: 0 / 1 = 0.00%
Accuracy: 1 / 2 = 50.00%


  1%|▏         | 3/205 [00:04<04:27,  1.33s/it]

Accuracy: 1 / 3 = 33.33%
Accuracy: 1 / 4 = 25.00%
Accuracy: 1 / 5 = 20.00%
Accuracy: 2 / 6 = 33.33%
Accuracy: 3 / 7 = 42.86%
Accuracy: 3 / 8 = 37.50%
Accuracy: 3 / 9 = 33.33%
Accuracy: 4 / 10 = 40.00%
Accuracy: 5 / 11 = 45.45%
Accuracy: 6 / 12 = 50.00%


  6%|▋         | 13/205 [00:05<00:50,  3.80it/s]

...

Accuracy: 117 / 200 = 58.50%
Accuracy: 118 / 201 = 58.71%
Accuracy: 119 / 202 = 58.91%


 99%|█████████▉| 203/205 [04:14<00:00,  2.57it/s]

Accuracy: 120 / 203 = 59.11%


100%|██████████| 205/205 [04:16<00:00,  1.25s/it]

Accuracy: 121 / 204 = 59.31%
Accuracy: 121 / 205 = 59.02%

✅ Accuracy: 121 / 205 = 59.02%
❌ Errors: 0

